# Upload Artifacts to S3
titanic_modeling.ipynb에서 생성된 output/{RUN_ID}/를 S3 모델 버킷에 업로드

In [1]:
import os
import boto3
import yaml
from pathlib import Path
from botocore.exceptions import ClientError

In [2]:
CONF_DIR = Path("conf")
OUTPUT_DIR = Path("output")

with open(CONF_DIR / "env.yml") as f:
    env_cfg = yaml.safe_load(f)

with open(CONF_DIR / "meta.yml") as f:
    meta_cfg = yaml.safe_load(f)

ENV          = env_cfg["env"]
REGION       = env_cfg["region"]
MODEL_BUCKET = env_cfg["s3"]["model_bucket"]
USER_ID      = meta_cfg["user_id"]
PROJECT      = meta_cfg["project"]
EXPERIMENT   = meta_cfg["experiment"]

In [5]:
# 자동 탐지: output/ 하위 디렉토리 중 가장 최근 것
run_dirs = sorted(
    [d for d in OUTPUT_DIR.iterdir() 
     if d.is_dir() and d.name != '.ipynb_checkpoints'],
    key=lambda d: d.stat().st_mtime,
    reverse=True
)

if not run_dirs:
    raise FileNotFoundError(f"No run directories found in {OUTPUT_DIR}")

print(f"Found run directories: {run_dirs}")
RUN_ID = run_dirs[0].name
LOCAL_OUTPUT_DIR = OUTPUT_DIR / RUN_ID
print(f"RUN_ID        : {RUN_ID}")
print(f"Local output  : {LOCAL_OUTPUT_DIR}")

Found run directories: [PosixPath('output/20260311_lightgbm_tuned_9fcc09e6')]
RUN_ID        : 20260311_lightgbm_tuned_9fcc09e6
Local output  : output/20260311_lightgbm_tuned_9fcc09e6


In [6]:
manifest_path = LOCAL_OUTPUT_DIR / "metadata" / "run_manifest.yml"
with open(manifest_path) as f:
    manifest = yaml.safe_load(f)

# run_manifest에 기록된 model_path 사용 (없으면 env.yml 패턴으로 직접 빌드)
if manifest.get("s3_refs", {}).get("model_path"):
    S3_URI   = manifest["s3_refs"]["model_path"].rstrip("/")
    s3_parts = S3_URI.replace("s3://", "").split("/", 1)
    S3_BUCKET, S3_PREFIX = s3_parts[0], s3_parts[1]
else:
    S3_PREFIX = f"{ENV}/{USER_ID}/{PROJECT}/{EXPERIMENT}/{RUN_ID}"
    S3_BUCKET = MODEL_BUCKET
    S3_URI    = f"s3://{S3_BUCKET}/{S3_PREFIX}"

print(f"S3 target     : {S3_URI}/")

S3 target     : s3://gs-retail-awesome-model-us-east-1/dev/hjsong/titanic-survival-prediction/tuned_v1/20260311_lightgbm_tuned_9fcc09e6/


In [7]:
s3_client = boto3.client("s3", region_name=REGION)

def upload_directory_tree(local_base, bucket, s3_prefix, dry_run=False):
    """디렉토리 트리 전체를 S3에 업로드 (run_pm_1/run_pm.ipynb 동일 로직)"""
    uploaded = []
    for root, dirs, files in os.walk(local_base):
        for filename in files:
            local_path = os.path.join(root, filename)
            relative_path = os.path.relpath(local_path, local_base)
            s3_key = f"{s3_prefix}/{relative_path}"
            s3_uri  = f"s3://{bucket}/{s3_key}"
            if dry_run:
                print(f"  [DRY RUN] {local_path} → {s3_uri}")
            else:
                s3_client.upload_file(local_path, bucket, s3_key)
                print(f"  ✓ Uploaded: {s3_uri}")
            uploaded.append(s3_uri)
    return uploaded

In [9]:
DRY_RUN = False   # True로 바꾸면 실제 업로드 없이 경로만 출력

print("=" * 70)
print("Upload Artifacts to S3")
print("=" * 70)
print(f"  Local : {LOCAL_OUTPUT_DIR}")
print(f"  S3    : {S3_URI}/")
print(f"  Dry   : {DRY_RUN}")
print("-" * 70)

uploaded = upload_directory_tree(
    local_base=str(LOCAL_OUTPUT_DIR),
    bucket=S3_BUCKET,
    s3_prefix=S3_PREFIX,
    dry_run=DRY_RUN
)

print("=" * 70)
print(f"완료: {len(uploaded)}개 파일 업로드됨")
print("=" * 70)

Upload Artifacts to S3
  Local : output/20260311_lightgbm_tuned_9fcc09e6
  S3    : s3://gs-retail-awesome-model-us-east-1/dev/hjsong/titanic-survival-prediction/tuned_v1/20260311_lightgbm_tuned_9fcc09e6/
  Dry   : False
----------------------------------------------------------------------
  ✓ Uploaded: s3://gs-retail-awesome-model-us-east-1/dev/hjsong/titanic-survival-prediction/tuned_v1/20260311_lightgbm_tuned_9fcc09e6/config/meta.yml
  ✓ Uploaded: s3://gs-retail-awesome-model-us-east-1/dev/hjsong/titanic-survival-prediction/tuned_v1/20260311_lightgbm_tuned_9fcc09e6/config/model.yml
  ✓ Uploaded: s3://gs-retail-awesome-model-us-east-1/dev/hjsong/titanic-survival-prediction/tuned_v1/20260311_lightgbm_tuned_9fcc09e6/config/env.yml
  ✓ Uploaded: s3://gs-retail-awesome-model-us-east-1/dev/hjsong/titanic-survival-prediction/tuned_v1/20260311_lightgbm_tuned_9fcc09e6/metadata/run_manifest.yml
  ✓ Uploaded: s3://gs-retail-awesome-model-us-east-1/dev/hjsong/titanic-survival-prediction/tuned_v

In [10]:
print(f"\n[업로드 결과 요약]")
print(f"  RUN_ID  : {RUN_ID}")
print(f"  S3 URI  : {S3_URI}/")
print(f"  파일 수  : {len(uploaded)}")


[업로드 결과 요약]
  RUN_ID  : 20260311_lightgbm_tuned_9fcc09e6
  S3 URI  : s3://gs-retail-awesome-model-us-east-1/dev/hjsong/titanic-survival-prediction/tuned_v1/20260311_lightgbm_tuned_9fcc09e6/
  파일 수  : 14
